# Production slim-ffill Ridge (S1 Equities)

Live-only full-sample retrain of the locked slim-ffill Ridge recipe.

- Features: same 20-col slim set as `05_training_linear_slim_ffill` (`FEATURE_REMOVE`)
- Panel: static 2015-frozen top-100 via `s1_engineered_features` + train panel calendar
- NaNs: unlimited within-ticker ffill, then complete-case drop
- **No** train-time CS re-rank (store `normalize=True` where supported)
- Alpha: frozen at **10.0** (no grid)
- Fit sample: IS train ∪ val ∪ post-IS holdout
- Artifact: `model_artifacts/s1_production_linear_model.joblib`

Do **not** use this model for OOS / holdout metric claims. Research scores remain in
`s1_linear_slim_ffill_is_predictions.parquet` from notebook 05.


## 0. Imports & Config


In [1]:
import os
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge

warnings.filterwarnings("ignore")

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from models.s1_equities.training_common import (
    LABEL_COL,
    TARGET_COL,
    add_cs_pct_target,
    chronological_is_split,
    default_paths,
    drop_nonfinite_labels,
    prepare_s1_week_panel,
)
from data.processing.cleaner import forward_fill_panel

PATHS = default_paths(ROOT)
FEATURES_PATH = PATHS["features"]
TRAIN_PANEL_PATH = PATHS["train_panel"]
MODEL_PATH = os.path.join(
    PATHS["model_dir"], "s1_production_linear_model.joblib"
)
os.makedirs(PATHS["model_dir"], exist_ok=True)

VAL_FRAC = 0.15
EMBARGO_WEEKS = 1
BEST_ALPHA = 10.0
RANDOM_SEED = 42

FEATURE_REMOVE = [
    "earnings_yield",
    "smart_beta_mom_126",
    "gdelt_abnormal_attention_5_60",
    "smart_beta_smb_84",
    "val_mom_resid_126_252_10",
    "obv_mom_soft_126_21_20",
    "near_52w_ratio_252_raw",
]

print(f"ROOT={ROOT}")
print(f"FEATURES_PATH={FEATURES_PATH}")
print(f"MODEL_PATH={MODEL_PATH}")
print(f"BEST_ALPHA={BEST_ALPHA}")
print(f"FEATURE_REMOVE requested ({len(FEATURE_REMOVE)}): {FEATURE_REMOVE}")


ROOT=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio
FEATURES_PATH=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_engineered_features.parquet
MODEL_PATH=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_artifacts\s1_production_linear_model.joblib
BEST_ALPHA=10.0
FEATURE_REMOVE requested (7): ['earnings_yield', 'smart_beta_mom_126', 'gdelt_abnormal_attention_5_60', 'smart_beta_smb_84', 'val_mom_resid_126_252_10', 'obv_mom_soft_126_21_20', 'near_52w_ratio_252_raw']


## 1. Load week panel + slim feature set

Uses `prepare_s1_week_panel`, then drops `FEATURE_REMOVE` names from the engineered set.


In [2]:
df, all_feats, _prices = prepare_s1_week_panel(FEATURES_PATH, TRAIN_PANEL_PATH)

missing = [c for c in FEATURE_REMOVE if c not in df.columns]
remove_resolved = [c for c in FEATURE_REMOVE if c in df.columns]
if missing:
    print(f"WARNING: FEATURE_REMOVE not in panel (skipped): {missing}")

FEATURE_COLS = [c for c in all_feats if c not in set(remove_resolved)]
print(f"FEATURE_REMOVE resolved ({len(remove_resolved)}): {remove_resolved}")
print(f"FEATURE_COLS kept ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(
    f"IS weeks={df.loc[df['is_research_is'], 'date'].nunique():,}  "
    f"holdout weeks={df.loc[~df['is_research_is'], 'date'].nunique():,}"
)
df.head()


FEATURE_REMOVE resolved (7): ['earnings_yield', 'smart_beta_mom_126', 'gdelt_abnormal_attention_5_60', 'smart_beta_smb_84', 'val_mom_resid_126_252_10', 'obv_mom_soft_126_21_20', 'near_52w_ratio_252_raw']
FEATURE_COLS kept (20): ['raw_momentum_252_5', 'smart_residual_mom_189_42', 'rel_downside_beta_252', 'rel_upside_beta_63', 'smart_beta_hml_252', 'downside_beta_42', 'upside_beta_42', 'size_mom_126', 'val_roc_pb_252', 'val_roc_pe_252', 'log_mcap', 'val_mom_dist_252_21', 'gross_profitability', 'filing_clock_expected_until', 'short_flow_ratio', 'market_corr', 'beta_mkt_interact', 'abnormal_volume', 'gdelt_tone_x_attention_21', 'gdelt_attention_5']
IS weeks=422  holdout weeks=443


,date,ticker,feature_date,open,high,low,close,volume,fwd_ret_1,fwd_ret_5,...,gross_profitability,filing_clock_expected_until,short_flow_ratio,market_corr,beta_mkt_interact,abnormal_volume,gdelt_tone_x_attention_21,gdelt_attention_5,gdelt_abnormal_attention_5_60,is_research_is
0,2010-01-05,AAPL,2010-01-04,6.424143,6.421146,6.357683,6.406478,493729600.0,-0.001025,-0.025210,...,NaN,27.0,0.439729,NaN,NaN,NaN,NaN,NaN,NaN,False
1,2010-01-05,ABT,2010-01-04,18.082116,18.111994,17.899535,18.078796,10829095.0,-0.009730,0.013402,...,NaN,31.0,0.255644,NaN,NaN,NaN,NaN,NaN,NaN,False
2,2010-01-05,ADBE,2010-01-04,37.040001,37.299999,36.650002,37.090000,4710200.0,0.007829,-0.024298,...,NaN,1.0,0.383420,NaN,NaN,NaN,NaN,NaN,NaN,False
3,2010-01-05,AET,2010-01-04,29.889494,30.016532,28.918587,29.943939,5671979.0,-0.013358,-0.009411,...,NaN,NaN,0.295322,NaN,NaN,NaN,NaN,NaN,NaN,False
4,2010-01-05,AIG,2010-01-04,18.609565,18.957173,18.255744,18.553696,7750900.0,-0.021014,-0.013009,...,NaN,4.0,0.417434,NaN,NaN,NaN,NaN,NaN,NaN,False


## 2. Forward-fill engineered features


In [3]:
_cov_before = {c: float(df[c].notna().mean()) for c in FEATURE_COLS}
df = forward_fill_panel(df, columns=FEATURE_COLS, limit=None)
print("FFILL coverage on FEATURE_COLS (week-start, before -> after):")
for c in FEATURE_COLS:
    after = float(df[c].notna().mean())
    print(
        f"  {c}: {100 * _cov_before[c]:.2f}% -> {100 * after:.2f}% "
        f"({100 * (after - _cov_before[c]):+.2f} pp)"
    )


FFILL coverage on FEATURE_COLS (week-start, before -> after):
  raw_momentum_252_5: 93.79% -> 93.79% (+0.00 pp)
  smart_residual_mom_189_42: 70.68% -> 70.68% (+0.00 pp)
  rel_downside_beta_252: 93.79% -> 93.79% (+0.00 pp)
  rel_upside_beta_63: 98.36% -> 98.36% (+0.00 pp)
  smart_beta_hml_252: 73.73% -> 73.73% (+0.00 pp)
  downside_beta_42: 68.86% -> 98.59% (+29.73 pp)
  upside_beta_42: 82.29% -> 98.83% (+16.54 pp)
  size_mom_126: 84.77% -> 88.80% (+4.03 pp)
  val_roc_pb_252: 71.06% -> 81.69% (+10.63 pp)
  val_roc_pe_252: 62.09% -> 87.71% (+25.62 pp)
  log_mcap: 88.25% -> 94.68% (+6.43 pp)
  val_mom_dist_252_21: 73.52% -> 83.67% (+10.15 pp)
  gross_profitability: 60.09% -> 60.09% (+0.00 pp)
  filing_clock_expected_until: 65.30% -> 94.26% (+28.96 pp)
  short_flow_ratio: 69.61% -> 99.18% (+29.57 pp)
  market_corr: 93.79% -> 93.79% (+0.00 pp)
  beta_mkt_interact: 93.79% -> 93.79% (+0.00 pp)
  abnormal_volume: 98.36% -> 98.36% (+0.00 pp)
  gdelt_tone_x_attention_21: 57.12% -> 63.33% (+6.21 

## 3. CS-rank target


In [4]:
df = add_cs_pct_target(df)
df = drop_nonfinite_labels(df, [LABEL_COL, TARGET_COL])
print(f"rows after label drop={len(df):,}  target={TARGET_COL}")


rows after label drop=85,179  target=fwd_ret_5_cs_pct


## 4. Engineered features + complete-case drop

No train-time CS re-rank. Drop any row still NaN on a model feature after ffill.


In [5]:
X_raw = df[FEATURE_COLS].copy()

nan_share = X_raw.isna().mean().sort_values(ascending=False)
print("Per-feature NaN share after ffill (before drop):")
display((100 * nan_share).rename("nan_pct").to_frame().round(2).head(20))

n_before = len(df)
dates_before = df["date"].nunique()
span_before = (df["date"].min(), df["date"].max())
nan_any = X_raw.isna().any(axis=1)
n_dropped = int(nan_any.sum())
df = df.loc[~nan_any].copy()
X = X_raw.loc[~nan_any].copy()
n_after = len(df)

print(
    "Complete-case drop (NaN in any model feature after ffill):\n"
    f"  rows:  before={n_before:,}  after={n_after:,}  "
    f"dropped={n_dropped:,} ({100 * n_dropped / max(n_before, 1):.2f}%)\n"
    f"  dates: before={dates_before:,}  after={df['date'].nunique():,}\n"
    f"  span:  before={span_before[0].date()} -> {span_before[1].date()}  "
    f"after={(df['date'].min().date() if n_after else 'n/a')} -> "
    f"{(df['date'].max().date() if n_after else 'n/a')}"
)
if n_after == 0:
    raise ValueError(
        "No rows left after complete-case drop; relax FEATURE_REMOVE or ffill coverage."
    )


Per-feature NaN share after ffill (before drop):


,nan_pct
gross_profitability,39.93
gdelt_tone_x_attention_21,36.70
gdelt_attention_5,34.20
smart_residual_mom_189_42,29.35
smart_beta_hml_252,26.30
val_roc_pb_252,18.32
val_mom_dist_252_21,16.35
val_roc_pe_252,12.30
size_mom_126,11.21
beta_mkt_interact,6.22


Complete-case drop (NaN in any model feature after ffill):
  rows:  before=85,179  after=35,082  dropped=50,097 (58.81%)
  dates: before=864  after=592
  span:  before=2010-01-05 -> 2026-07-20  after=2015-03-23 -> 2026-07-20


## 5. Chronological split (diagnostics) + production fit sample

Alpha is frozen. Split reports train / val / holdout sizes; fit uses
IS train ∪ val ∪ post-IS holdout.


In [6]:
split = chronological_is_split(
    df, val_frac=VAL_FRAC, embargo_weeks=EMBARGO_WEEKS
)
train_df, val_df, holdout_df = split.train_df, split.val_df, split.holdout_df
for label, part in (("train", train_df), ("val", val_df), ("holdout", holdout_df)):
    print(
        f"{label}: rows={len(part):,}  weeks={part['date'].nunique() if len(part) else 0}"
    )

prod_df = pd.concat([train_df, val_df, holdout_df], axis=0).sort_values(
    ["date", "ticker"]
)
X_prod = X.loc[prod_df.index]
y_prod = prod_df[TARGET_COL]
print(
    f"Production fit sample: rows={len(prod_df):,}  "
    f"weeks={prod_df['date'].nunique():,}  "
    f"{prod_df['date'].min().date()} -> {prod_df['date'].max().date()}"
)


train: rows=18,827  weeks=348
val: rows=4,030  weeks=62
holdout: rows=12,160  weeks=181
Production fit sample: rows=35,017  weeks=591  2015-03-23 -> 2026-07-20


## 6. Fit Ridge (alpha=10) + dump joblib

Live only — do not rewrite OOS IC claims from this fit.


In [7]:
model = Ridge(alpha=BEST_ALPHA, random_state=RANDOM_SEED)
model.fit(X_prod, y_prod)

payload = {
    "model": model,
    "feature_cols": FEATURE_COLS,
    "feature_remove": remove_resolved,
    "alpha": BEST_ALPHA,
    "label_col": LABEL_COL,
    "target_col": TARGET_COL,
    "random_seed": RANDOM_SEED,
    "fit_rows": int(len(prod_df)),
    "fit_date_min": str(prod_df["date"].min().date()),
    "fit_date_max": str(prod_df["date"].max().date()),
}
joblib.dump(payload, MODEL_PATH)
print(f"Saved production model (live only): {MODEL_PATH}")
print(f"  alpha={BEST_ALPHA}  n_features={len(FEATURE_COLS)}  n_rows={len(prod_df):,}")

coef = (
    pd.Series(model.coef_, index=FEATURE_COLS)
    .sort_values(key=np.abs, ascending=False)
    .rename("coef")
)
display(coef.to_frame())


Saved production model (live only): c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_artifacts\s1_production_linear_model.joblib
  alpha=10.0  n_features=20  n_rows=35,017


,coef
beta_mkt_interact,-0.080908
smart_residual_mom_189_42,0.078985
smart_beta_hml_252,-0.024007
upside_beta_42,0.019176
log_mcap,-0.015045
val_mom_dist_252_21,-0.011333
rel_downside_beta_252,0.011026
size_mom_126,-0.010160
downside_beta_42,0.006759
market_corr,0.004363
